# Enterprise Q — Text-to-SQL QLoRA Fine-Tune

Fine-tunes **Qwen2.5-Coder-1.5B-Instruct** with QLoRA on `b-mc2/sql-create-context`,
using the **exact prompt template Enterprise Q uses at runtime**.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Run All (~1 hour).

Outputs: `sql_lora_adapter/` (LoRA weights) and optionally a GGUF for Ollama/llama.cpp.

In [ ]:
# Install Unsloth (brings torch/transformers/peft/trl/bitsandbytes at compatible versions).
# If this cell errors on a fresh Colab image, check https://docs.unsloth.ai for the current install line.
%pip install -q unsloth


In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",  # 4-bit NF4 = QLoRA
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()


In [ ]:
from datasets import load_dataset

# Same template as backend/utils/prompt_templates.py :: SQL_GENERATION_PROMPT —
# training on the exact runtime prompt avoids train/serve skew.
PROMPT = """You are an expert SQL analyst. Given the database schema below,
write a single SQLite-compatible SELECT query that answers the user's question.

**Rules:**
- Output ONLY the SQL query — no explanation, no markdown.
- Use only columns and tables that exist in the schema.
- Never use DROP, DELETE, UPDATE, INSERT, ALTER, or CREATE.
- If the question is ambiguous, make a reasonable assumption.

Schema:
{schema}

User question: {question}

SQL Query:"""

N_TRAIN, N_EVAL = 10_000, 200

raw = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=42)
raw = raw.select(range(N_TRAIN + N_EVAL))

def to_chat(ex):
    messages = [
        {"role": "user", "content": PROMPT.format(schema=ex["context"], question=ex["question"])},
        {"role": "assistant", "content": ex["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_ds = raw.select(range(N_TRAIN)).map(to_chat, remove_columns=raw.column_names)
eval_raw = raw.select(range(N_TRAIN, N_TRAIN + N_EVAL))  # held out, never trained on
print(train_ds[0]["text"][:600])


In [ ]:
import re

def normalize_sql(s):
    s = re.sub(r"```(?:sql)?|```", "", s)
    return re.sub(r"\s+", " ", s).strip().rstrip(";").lower()

def evaluate(model, tokenizer, examples, label):
    FastLanguageModel.for_inference(model)
    hits = 0
    for ex in examples:
        messages = [{"role": "user", "content": PROMPT.format(schema=ex["context"], question=ex["question"])}]
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
        out = model.generate(input_ids=inputs, max_new_tokens=128, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        pred = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
        hits += normalize_sql(pred) == normalize_sql(ex["answer"])
    acc = hits / len(examples)
    print(f"{label}: {hits}/{len(examples)} = {acc:.1%} exact match")
    return acc

# LoRA B-matrices start at zero, so before training the adapter is a no-op —
# this measures the UNTUNED base model on the held-out split.
base_acc = evaluate(model, tokenizer, eval_raw.select(range(100)), "BASE (no tuning)")


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        output_dir="outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # effective batch = 16
        num_train_epochs=1,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        warmup_ratio=0.05,
        logging_steps=25,
        optim="adamw_8bit",
        seed=42,
        fp16=True,
        report_to="none",
    ),
)
trainer.train()


In [ ]:
# Post-training accuracy on the SAME held-out examples
tuned_acc = evaluate(model, tokenizer, eval_raw.select(range(100)), "TUNED (+LoRA)")
print(f"\nImprovement: {base_acc:.1%} -> {tuned_acc:.1%}")


In [ ]:
# Save the adapter (~70 MB) — download this folder from the Files panel
model.save_pretrained("sql_lora_adapter")
tokenizer.save_pretrained("sql_lora_adapter")

# Optional: push to your Hugging Face account (uncomment + add your token)
# model.push_to_hub("<your-hf-username>/enterprise-q-sql-lora", token="hf_...")

# Optional: merged GGUF for Ollama / llama.cpp serving (adds ~10 min)
# model.save_pretrained_gguf("sql_lora_gguf", tokenizer, quantization_method="q4_k_m")
print("Saved to sql_lora_adapter/")


In [ ]:
# Quick demo on an Enterprise Q-style schema
demo_schema = """CREATE TABLE employees (employee_id INTEGER, employee_name TEXT, department_id INTEGER, salary_usd INTEGER, attrition TEXT);
CREATE TABLE departments (department_id INTEGER, department_name TEXT, location TEXT, annual_budget_usd INTEGER);"""

messages = [{"role": "user", "content": PROMPT.format(
    schema=demo_schema,
    question="Which department has the highest average salary? Show the department name.")}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))
